# 6. Spectroscopy Extraction

In this tutorial, you will learn how to run the Spectroscopy Pipeline to extract 1D spectral data from 2D grating dispersion images.


## 6.1 Providing a Master Frame

`spectral_stacking.stacked_image` behaves the same way as `stacking.stacked_image` did in Section 4.1. It is an ordinary field on the `Target` object, allowing the target to hold both its imaging stack (`stacking.stacked_image`) and its dispersion stack (`spectral_stacking.stacked_image`). The spectroscopy stage of `processing.process_target` does not require it to have come from `processing.stack()`—setting it yourself is useful when you already have a master frame from elsewhere. Here we point it at one bundled with the sample data for **M 13**.


In [1]:
from astrometricslib import Astrometrics
from documentation.notebooks.astrometrics.scripts.sample_data_staging import stage_m13_sample_data

astrometrics = Astrometrics()
target = astrometrics.targets.get("M 13")

if target:
    sample_data_dir = stage_m13_sample_data()
    target.spectral_stacking.stacked_image = str(
        sample_data_dir / "M_13_SPEC_Stacked_ZWO_ASI_533MM_Pro_405mm.fits"
    )
    astrometrics.targets.save()
    print(f"target.spectral_stacking.stacked_image = {target.spectral_stacking.stacked_image}")
else:
    print("Target not found.")

target.spectral_stacking.stacked_image = /home/user/astrometrics/libraryIndex/sample_data_working/M 13/M_13_SPEC_Stacked_ZWO_ASI_533MM_Pro_405mm.fits


## 6.2 Running the Spectroscopy Pipeline

We use `astrometrics.processing.process_target(target, stages=["spectroscopy"])` on the target. It automatically detects the spectroscopic master frame (`spectral_stacking.stacked_image`) or raw spectroscopic frames (denoted by the 'SPEC' filter).

:::{note}
**Flux Calibration Status.** The extracted 1D spectrum yields relative flux counts (instrumental ADU) along a calibrated wavelength axis (in Ångströms). Converting this to absolute energy flux ($F_\lambda$) requires an instrument response curve derived from a spectrophotometric standard star to remove sensor quantum efficiency and grating throughput variations.
:::

In [2]:
print("Starting spectroscopy pipeline...")
if target:
    result = astrometrics.processing.process_target(target, stages=["spectroscopy"]).results["spectroscopy"]
    summary = target.quality.spectroscopy
    failed_gates = [gate.name for gate in summary.gates if gate.status.value == "failed"] if summary else []
    status = "failed gates: " + ", ".join(failed_gates) if failed_gates else "completed"
    print(f"Spectroscopy Pipeline Status: {status}")
    stars = result.get("stellar_objects", [])
    print(f"Spectra extracted: {len(stars)}")

    # Each star keeps its match score and its gap to another class.
    print("\nSelf-determined spectral types (lower match score is closer):")
    for star in stars:
        spectrum = astrometrics.stars.get_spectrum(star.id)
        if spectrum is not None and spectrum.self_determined_spectral_type_rms is not None:
            print(
                f"  {star.id}: {spectrum.self_determined_spectral_type} "
                f"(match score={spectrum.self_determined_spectral_type_rms:.3f}, "
                f"gap to next class={spectrum.rms_gap_to_next_class})"
            )
else:
    print("Target not found.")

Starting spectroscopy pipeline...


The current archive release 3.15 contains two breaking changes. Please see https://www.cosmos.esa.int/web/gaia/news#GACSrelease315 for details. In case of questions or feedback, please consult the Gaia helpdesk at https://www.cosmos.esa.int/web/gaia/gaia-helpdesk.


Spectroscopy Pipeline Status: failed gates: spectral_classification, feature_significance, processing_quality, wavelength_scale, gaia_xp_agreement
Spectra extracted: 424

Self-determined spectral types (lower match score is closer):
  HD 150998: K5V (match score=0.136, gap to next class=0.040467883603420574)
  HD 150679: A7V (match score=0.052, gap to next class=0.02828826993886603)
  Gaia DR3 1327617929278818176: K7V (match score=0.138, gap to next class=0.01842270176196356)
  HD 150293: K2V (match score=0.076, gap to next class=0.044453935988018875)
  HD 151023: K2V (match score=0.046, gap to next class=0.05768715937319331)
  BD+36  2775: K5V (match score=0.122, gap to next class=0.08358541599846468)
  HD 151086: K2V (match score=0.094, gap to next class=0.05236026325145586)
  BD+35  2851: K7V (match score=0.148, gap to next class=0.009280185275630404)
  BD+36  2764: K0V (match score=0.099, gap to next class=0.008446383573120997)
  TYC 2588-1167-1: K4V (match score=0.131, gap to next

  NGC  6205   187: K2V (match score=0.160, gap to next class=0.017063892538982967)
  NGC  6205   954: M2V (match score=0.306, gap to next class=0.08368321082249874)
  NGC  6205   523: K0V (match score=0.156, gap to next class=0.00867349459233796)
  NGC  6205   488: F0V (match score=0.119, gap to next class=0.03179697679073763)
  NGC  6205   261: F8V (match score=0.105, gap to next class=0.0089477955659589)
  NGC  6205   367: K0V (match score=0.162, gap to next class=0.007163090747833195)
  NGC  6205   786: M0V (match score=0.199, gap to next class=0.007119692959692614)
  NGC  6205   832: K0V (match score=0.134, gap to next class=0.007647714070954131)
  NGC  6205   414: M0V (match score=0.227, gap to next class=0.004485187044721228)
  NGC  6205   352: M0V (match score=0.195, gap to next class=0.004287544406946481)
  NGC  6205   384: F6V (match score=0.171, gap to next class=0.011965609570136093)
  NGC  6205  1119: G2V (match score=0.185, gap to next class=0.0077399085933585465)
  NGC  6

  FIELD_J250.5039+36.3966: M4V (match score=0.310, gap to next class=0.3138262278684654)
  Gaia DR3 1328056874936120192: M0V (match score=0.219, gap to next class=0.00444820413227931)
  NGC  6205   775: M3V (match score=0.335, gap to next class=0.17583211630192935)
  UCAC4 633-055266: M5V (match score=0.438, gap to next class=0.4496599780687041)
  2MASS J16414398+3622338: M5V (match score=0.431, gap to next class=0.47562494521621307)
  NGC  6205   848: M2V (match score=0.308, gap to next class=0.10947961858258043)
  NGC  6205   114: M0V (match score=0.181, gap to next class=0.010846874169168713)
  FIELD_J250.3835+36.4246: M2V (match score=0.237, gap to next class=0.10341152105958062)
  NGC  6205   169: M2V (match score=0.195, gap to next class=0.09532062454729362)
  NGC  6205   991: M4V (match score=0.424, gap to next class=0.2849037898444818)


  2MASS J16413082+3630130: A3V (match score=0.115, gap to next class=0.013330674430027806)
  NGC  6205   439: G0V (match score=0.119, gap to next class=0.0018830149850733585)
  GPM 250.789509+36.475354: M5V (match score=0.427, gap to next class=0.4810777049933058)
  NGC  6205   638: M0V (match score=0.234, gap to next class=0.007402695326455572)
  FIELD_J250.4398+36.3886: M4V (match score=0.393, gap to next class=0.39976686935779593)
  NGC  6205   117: K0V (match score=0.186, gap to next class=0.013737214541196141)
  NGC  6205    96: K3V (match score=0.188, gap to next class=0.05487232187562813)


  NGC  6205   955: M3V (match score=0.340, gap to next class=0.153090258923607)
  2MASS J16413013+3626354: M1V (match score=0.197, gap to next class=0.03893533258992857)
  2MASS J16425528+3636561: O5V (match score=0.205, gap to next class=0.013210472470104456)
  2MASS J16413862+3630369: A5V (match score=0.141, gap to next class=0.013163640134542853)
  FIELD_J250.3922+36.3933: M4V (match score=0.307, gap to next class=0.36840308978597575)
  NGC  6205   833: M4V (match score=0.381, gap to next class=0.31819536602488646)
